# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AxelYoel/FlyRank-AI-Internship---Axel-Yoel-Chandra/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window


One row equals one page (content_hash_id), aggregated over the month of 2026-03. It is built by summing daily impressions/clicks and taking an impression-weighted average of the avg position across the month. The visibility floor is pages with ≥50 impressions — directly tested by confirming the top 20 ranked pages by lost_clicks all had 60k+ impressions, showing the floor doesn't distort the ranking.

I use 2026-03, the mid-panel month, not June, the sealed final month, because if I build or design my label logic using June data, my model would be built and shaped by the answer or the future outcome that I want to predict, which would make any future evaluation against it meaningless.



In [1]:
%pip -q install duckdb
import duckdb
con = duckdb.connect()

import duckdb, os
from google.colab import userdata

hf_token = userdata.get("HF_token")  # exact name you set in Colab Secrets
os.environ["HF_token"] = hf_token

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

In [2]:
from huggingface_hub import HfApi
api = HfApi()
files = api.list_repo_files("FlyRank/internship-warehouse", repo_type="dataset")
[f for f in files if "daily_performance" in f]

['fact_content_daily_performance/month=2025-01/data_0.parquet',
 'fact_content_daily_performance/month=2025-02/data_0.parquet',
 'fact_content_daily_performance/month=2025-03/data_0.parquet',
 'fact_content_daily_performance/month=2025-04/data_0.parquet',
 'fact_content_daily_performance/month=2025-05/data_0.parquet',
 'fact_content_daily_performance/month=2025-06/data_0.parquet',
 'fact_content_daily_performance/month=2025-07/data_0.parquet',
 'fact_content_daily_performance/month=2025-08/data_0.parquet',
 'fact_content_daily_performance/month=2025-09/data_0.parquet',
 'fact_content_daily_performance/month=2025-10/data_0.parquet',
 'fact_content_daily_performance/month=2025-11/data_0.parquet',
 'fact_content_daily_performance/month=2025-12/data_0.parquet',
 'fact_content_daily_performance/month=2026-01/data_0.parquet',
 'fact_content_daily_performance/month=2026-02/data_0.parquet',
 'fact_content_daily_performance/month=2026-03/data_0.parquet',
 'fact_content_daily_performance/month=2

In [3]:
con.sql(f"""
    DESCRIBE SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
""").df()



,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [4]:
con.sql(f"""
    SELECT COUNT(*) AS n_rows, MIN(report_date) AS earliest, MAX(report_date) AS latest
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_rows,earliest,latest
0,9841378,2026-03-01,2026-03-31


In [5]:
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{rel}/dim_content.parquet')").df()

,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,keyword_hash_id,VARCHAR,YES,None,None,None
3,url_hash_id,VARCHAR,YES,None,None,None
4,keyword_char_count,BIGINT,YES,None,None,None
5,keyword_token_count,BIGINT,YES,None,None,None
6,url_char_count,BIGINT,YES,None,None,None
7,content_created_date,DATE,YES,None,None,None
8,content_updated_date,DATE,YES,None,None,None
9,content_type,VARCHAR,YES,None,None,None


In [6]:
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{rel}/fact_content_query_90d.parquet')").df()

,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,query_hash_id,VARCHAR,YES,None,None,None
3,query_char_count,BIGINT,YES,None,None,None
4,query_token_count,BIGINT,YES,None,None,None
5,window_start,DATE,YES,None,None,None
6,window_end,DATE,YES,None,None,None
7,impressions_90d,BIGINT,YES,None,None,None
8,clicks_90d,BIGINT,YES,None,None,None
9,impressions_last30,BIGINT,YES,None,None,None


In [7]:
con.sql(f"""
    SELECT MIN(window_start), MAX(window_start), MIN(window_end), MAX(window_end)
    FROM read_parquet('{rel}/fact_content_query_90d.parquet')
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,min(window_start),max(window_start),min(window_end),max(window_end)
0,2026-04-02,2026-04-02,2026-06-30,2026-06-30


In [8]:
con.sql(f"""
    SELECT
      COUNT(*) AS n_rows,
      SUM(CASE WHEN gsc_data_available THEN 1 ELSE 0 END) AS n_available_true,
      SUM(CASE WHEN gsc_data_available IS NULL THEN 1 ELSE 0 END) AS n_available_null,
      SUM(CASE WHEN gsc_impressions IS NULL THEN 1 ELSE 0 END) AS null_impressions,
      SUM(CASE WHEN gsc_clicks IS NULL THEN 1 ELSE 0 END) AS null_clicks,
      SUM(CASE WHEN gsc_avg_position IS NULL THEN 1 ELSE 0 END) AS null_position,
      MIN(gsc_avg_position) AS min_pos, MAX(gsc_avg_position) AS max_pos,
      MIN(gsc_impressions) AS min_impr, MAX(gsc_impressions) AS max_impr,
      SUM(CASE WHEN gsc_impressions = 0 THEN 1 ELSE 0 END) AS n_zero_impressions,
      SUM(CASE WHEN gsc_avg_position = 0 THEN 1 ELSE 0 END) AS n_zero_position
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_rows,n_available_true,n_available_null,null_impressions,null_clicks,null_position,min_pos,max_pos,min_impr,max_impr,n_zero_impressions,n_zero_position
0,9841378,3611061.0,0.0,0.0,0.0,6230317.0,0.0,498.0,0,40084,6230317.0,163189.0


In [9]:
con.sql(f"""
    SELECT gsc_data_available,
           COUNT(*) AS n_rows,
           SUM(CASE WHEN gsc_impressions = 0 THEN 1 ELSE 0 END) AS n_zero_impr,
           SUM(CASE WHEN gsc_avg_position IS NULL THEN 1 ELSE 0 END) AS n_null_pos
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
    GROUP BY 1
""").df()

,gsc_data_available,n_rows,n_zero_impr,n_null_pos
0,False,6230317,6230317.0,6230317.0
1,True,3611061,0.0,0.0


In [10]:
con.sql(f"""
    SELECT gsc_data_available, gsc_impressions, gsc_clicks, COUNT(*) AS n_rows
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
    WHERE gsc_avg_position = 0
    GROUP BY 1, 2, 3
    ORDER BY n_rows DESC
    LIMIT 10
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,gsc_data_available,gsc_impressions,gsc_clicks,n_rows
0,True,1,0,83931
1,True,2,0,31014
2,True,3,0,15637
3,True,4,0,8995
4,True,5,0,5791
5,True,6,0,3849
6,True,7,0,2670
7,True,8,0,1930
8,True,9,0,1419
9,True,10,0,1114


In [11]:
con.sql(f"""
    SELECT
      CASE WHEN gsc_clicks = 0 THEN 'zero_clicks' ELSE 'has_clicks' END AS click_bucket,
      COUNT(*) AS n_rows,
      SUM(CASE WHEN gsc_avg_position = 0 THEN 1 ELSE 0 END) AS n_position_zero
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
    WHERE gsc_data_available = TRUE
    GROUP BY 1
""").df()

,click_bucket,n_rows,n_position_zero
0,zero_clicks,3193080,162038.0
1,has_clicks,417981,1151.0


In [12]:
con.sql(f"""
    SELECT
      CASE WHEN gsc_avg_position = 0 THEN '0 (unknown meaning)'
           WHEN gsc_avg_position <= 3 THEN '1-3'
           WHEN gsc_avg_position <= 10 THEN '4-10'
           ELSE '11+' END AS pos_bucket,
      SUM(gsc_clicks) AS total_clicks,
      SUM(gsc_impressions) AS total_impressions,
      SUM(gsc_clicks)::DOUBLE / NULLIF(SUM(gsc_impressions), 0) AS ctr
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
    WHERE gsc_data_available = TRUE
    GROUP BY 1
    ORDER BY 1
""").df()

,pos_bucket,total_clicks,total_impressions,ctr
0,0 (unknown meaning),1174.0,468746.0,0.002505
1,1-3,204283.0,53559848.0,0.003814
2,11+,170547.0,88798882.0,0.001921
3,4-10,445828.0,137830113.0,0.003235


In [13]:
con.sql(f"""
    WITH daily AS (
        SELECT content_hash_id, gsc_avg_position, gsc_impressions, gsc_clicks
        FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
        WHERE gsc_data_available = TRUE
          AND gsc_avg_position > 0        -- excludes the sentinel rows we confirmed earlier
          AND gsc_impressions > 0
    )
    SELECT
        content_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,
        SUM(gsc_clicks)::DOUBLE / SUM(gsc_impressions) AS ctr,
        SUM(gsc_avg_position * gsc_impressions) / SUM(gsc_impressions) AS weighted_avg_position
    FROM daily
    GROUP BY content_hash_id
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,impressions,clicks,ctr,weighted_avg_position
0,content_7a105f548d9c6916,6523.0,7.0,0.001073,6.893301
1,content_a3ea9792f793ec72,424.0,0.0,0.000000,3.433962
2,content_36c36abc7650d7af,5630.0,6.0,0.001066,6.535346
3,content_a7da352b73b02668,4944.0,13.0,0.002629,7.435680
4,content_1855a661b4d36130,417.0,1.0,0.002398,3.983213
...,...,...,...,...,...
175299,content_4a46b45b40a57f11,1.0,0.0,0.000000,47.000000
175300,content_9eeeb3f77716113c,1.0,0.0,0.000000,9.000000
175301,content_2d8e52b436a736c8,3.0,0.0,0.000000,7.333333
175302,content_4d0dafdb2450a480,15.0,0.0,0.000000,42.466667


In [14]:
con.sql(f"""
    WITH daily AS (
        SELECT content_hash_id, gsc_avg_position, gsc_impressions, gsc_clicks
        FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
        WHERE gsc_data_available = TRUE AND gsc_avg_position > 0 AND gsc_impressions > 0
    ),
    rollup AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS impressions
        FROM daily GROUP BY content_hash_id
    )
    SELECT
        quantile_cont(impressions, 0.10) AS p10,
        quantile_cont(impressions, 0.25) AS p25,
        quantile_cont(impressions, 0.50) AS median,
        quantile_cont(impressions, 0.75) AS p75,
        quantile_cont(impressions, 0.90) AS p90,
        COUNT(*) AS n_pages
    FROM rollup
""").df()

,p10,p25,median,p75,p90,n_pages
0,3.0,18.0,172.0,1052.0,3962.0,175304


In [15]:
con.sql(f"""
    WITH daily AS (
        SELECT content_hash_id, gsc_avg_position, gsc_impressions, gsc_clicks
        FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
        WHERE gsc_data_available = TRUE AND gsc_avg_position > 0 AND gsc_impressions > 0
    ),
    rollup AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS impressions
        FROM daily GROUP BY content_hash_id
    )
    SELECT
        SUM(CASE WHEN impressions < 50 THEN 1 ELSE 0 END) AS below_50,
        SUM(CASE WHEN impressions < 100 THEN 1 ELSE 0 END) AS below_100,
        SUM(CASE WHEN impressions < 500 THEN 1 ELSE 0 END) AS below_500,
        COUNT(*) AS total
    FROM rollup
""").df()

,below_50,below_100,below_500,total
0,60857.0,75140.0,113612.0,175304


In [16]:
con.sql(f"""
    WITH daily AS (
        SELECT content_hash_id, gsc_avg_position, gsc_impressions, gsc_clicks
        FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
        WHERE gsc_data_available = TRUE AND gsc_avg_position > 0 AND gsc_impressions > 0
    ),
    rollup AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS impressions, SUM(gsc_clicks) AS clicks
        FROM daily GROUP BY content_hash_id
    )
    SELECT
        CASE WHEN impressions < 50 THEN 'under_50' ELSE '50_plus' END AS bucket,
        COUNT(*) AS n_pages,
        MIN(impressions) AS min_impr, MAX(impressions) AS max_impr
    FROM rollup
    GROUP BY 1
""").df()

,bucket,n_pages,min_impr,max_impr
0,50_plus,114447,50.0,617124.0
1,under_50,60857,1.0,49.0


In [17]:
con.sql(f"""
    WITH daily AS (
        SELECT content_hash_id, gsc_avg_position, gsc_impressions, gsc_clicks
        FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
        WHERE gsc_data_available = TRUE AND gsc_avg_position > 0 AND gsc_impressions > 0
    ),
    rollup AS (
        SELECT
            content_hash_id,
            SUM(gsc_impressions) AS impressions,
            SUM(gsc_clicks) AS clicks,
            SUM(gsc_clicks)::DOUBLE / SUM(gsc_impressions) AS ctr,
            SUM(gsc_avg_position * gsc_impressions) / SUM(gsc_impressions) AS weighted_avg_position
        FROM daily
        GROUP BY content_hash_id
        HAVING SUM(gsc_impressions) >= 50
    )
    SELECT
        *,
        CASE
            WHEN weighted_avg_position <= 3 THEN 'tier_1_1-3'
            WHEN weighted_avg_position <= 10 THEN 'tier_2_4-10'
            WHEN weighted_avg_position <= 20 THEN 'tier_3_11-20'
            ELSE 'tier_4_21plus'
        END AS position_tier
    FROM rollup
""").df()

,content_hash_id,impressions,clicks,ctr,weighted_avg_position,position_tier
0,content_b7e512995f79d5a6,1140.0,2.0,0.001754,4.450877,tier_2_4-10
1,content_905aa32a0230694e,132.0,0.0,0.000000,6.363636,tier_2_4-10
2,content_05434271b257bb68,1421.0,6.0,0.004222,6.906404,tier_2_4-10
3,content_d056587ff7faca0c,2770.0,16.0,0.005776,3.950542,tier_2_4-10
4,content_2662845f598544ef,142.0,1.0,0.007042,7.929577,tier_2_4-10
...,...,...,...,...,...,...
114442,content_74737bd274fce685,55.0,0.0,0.000000,6.636364,tier_2_4-10
114443,content_9605cc5c081a0a9f,158.0,1.0,0.006329,5.094937,tier_2_4-10
114444,content_de26b1f2e69bb2ca,225.0,0.0,0.000000,10.573333,tier_3_11-20
114445,content_f0553bcb1e781342,64.0,0.0,0.000000,2.390625,tier_1_1-3


In [18]:
con.sql(f"""
    WITH daily AS (
        SELECT content_hash_id, gsc_avg_position, gsc_impressions, gsc_clicks
        FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
        WHERE gsc_data_available = TRUE AND gsc_avg_position > 0 AND gsc_impressions > 0
    ),
    rollup AS (
        SELECT
            content_hash_id,
            SUM(gsc_impressions) AS impressions,
            SUM(gsc_clicks) AS clicks,
            SUM(gsc_avg_position * gsc_impressions) / SUM(gsc_impressions) AS weighted_avg_position
        FROM daily
        GROUP BY content_hash_id
        HAVING SUM(gsc_impressions) >= 50
    ),
    tiered AS (
        SELECT *,
            CASE
                WHEN weighted_avg_position <= 3 THEN 'tier_1_1-3'
                WHEN weighted_avg_position <= 10 THEN 'tier_2_4-10'
                WHEN weighted_avg_position <= 20 THEN 'tier_3_11-20'
                ELSE 'tier_4_21plus'
            END AS position_tier
        FROM rollup
    )
    SELECT position_tier, COUNT(*) AS n_pages, SUM(impressions) AS total_impressions
    FROM tiered
    GROUP BY 1
    ORDER BY 1
""").df()

,position_tier,n_pages,total_impressions
0,tier_1_1-3,10060,41114243.0
1,tier_2_4-10,52443,147581243.0
2,tier_3_11-20,22070,31025597.0
3,tier_4_21plus,29874,59672097.0


In [19]:
con.sql(f"""
    WITH daily AS (
        SELECT content_hash_id, gsc_avg_position, gsc_impressions, gsc_clicks
        FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
        WHERE gsc_data_available = TRUE AND gsc_avg_position > 0 AND gsc_impressions > 0
    ),
    rollup AS (
        SELECT
            content_hash_id,
            SUM(gsc_impressions) AS impressions,
            SUM(gsc_clicks) AS clicks,
            SUM(gsc_clicks)::DOUBLE / SUM(gsc_impressions) AS ctr,
            SUM(gsc_avg_position * gsc_impressions) / SUM(gsc_impressions) AS weighted_avg_position
        FROM daily
        GROUP BY content_hash_id
        HAVING SUM(gsc_impressions) >= 50
    ),
    tiered AS (
        SELECT *,
            CASE
                WHEN weighted_avg_position <= 3 THEN 'tier_1_1-3'
                WHEN weighted_avg_position <= 10 THEN 'tier_2_4-10'
                WHEN weighted_avg_position <= 20 THEN 'tier_3_11-20'
                ELSE 'tier_4_21plus'
            END AS position_tier
        FROM rollup
    ),
    ordered AS (
        SELECT
            position_tier, ctr, impressions,
            SUM(impressions) OVER (PARTITION BY position_tier ORDER BY ctr
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_impr,
            SUM(impressions) OVER (PARTITION BY position_tier) AS tier_total_impr
        FROM tiered
    )
    SELECT position_tier, MIN(ctr) AS weighted_median_ctr
    FROM ordered
    WHERE running_impr >= 0.5 * tier_total_impr
    GROUP BY position_tier
    ORDER BY position_tier
""").df()

,position_tier,weighted_median_ctr
0,tier_1_1-3,0.002811
1,tier_2_4-10,0.002353
2,tier_3_11-20,0.002226
3,tier_4_21plus,0.000748


In [20]:
con.sql(f"""
    WITH daily AS (
        SELECT content_hash_id, gsc_avg_position, gsc_impressions, gsc_clicks
        FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
        WHERE gsc_data_available = TRUE AND gsc_avg_position > 0 AND gsc_impressions > 0
    ),
    rollup AS (
        SELECT
            content_hash_id,
            SUM(gsc_impressions) AS impressions,
            SUM(gsc_clicks) AS clicks,
            SUM(gsc_avg_position * gsc_impressions) / SUM(gsc_impressions) AS weighted_avg_position
        FROM daily
        GROUP BY content_hash_id
        HAVING SUM(gsc_impressions) >= 50
    ),
    tiered AS (
        SELECT *,
            CASE
                WHEN weighted_avg_position <= 3 THEN 'tier_1_1-3'
                WHEN weighted_avg_position <= 10 THEN 'tier_2_4-10'
                WHEN weighted_avg_position <= 20 THEN 'tier_3_11-20'
                ELSE 'tier_4_21plus'
            END AS position_tier
        FROM rollup
    ),
    benchmark(position_tier, expected_ctr) AS (
        VALUES
            ('tier_1_1-3', 0.002811),
            ('tier_2_4-10', 0.002353),
            ('tier_3_11-20', 0.002226),
            ('tier_4_21plus', 0.000748)
    )
    SELECT
        t.content_hash_id, t.position_tier, t.impressions, t.clicks,
        b.expected_ctr,
        b.expected_ctr * t.impressions AS expected_clicks,
        (b.expected_ctr * t.impressions) - t.clicks AS lost_clicks
    FROM tiered t
    JOIN benchmark b USING (position_tier)
    ORDER BY lost_clicks DESC
    LIMIt 20
""").df()

,content_hash_id,position_tier,impressions,clicks,expected_ctr,expected_clicks,lost_clicks
0,content_44f34c0a90047651,tier_1_1-3,212404.0,24.0,0.002811,597.067644,573.067644
1,content_8e1334d6356668e3,tier_1_1-3,134984.0,1.0,0.002811,379.440024,378.440024
2,content_fec55986a1868d62,tier_1_1-3,124075.0,1.0,0.002811,348.774825,347.774825
3,content_34a70fea29d15f24,tier_2_4-10,143019.0,43.0,0.002353,336.523707,293.523707
4,content_8d7d99f109e19aa2,tier_1_1-3,203497.0,289.0,0.002811,572.030067,283.030067
5,content_f6116743b00afc2d,tier_2_4-10,107584.0,15.0,0.002353,253.145152,238.145152
6,content_9c057b66c30a3abb,tier_1_1-3,83832.0,1.0,0.002811,235.651752,234.651752
7,content_7c6373141eae744a,tier_2_4-10,132593.0,83.0,0.002353,311.991329,228.991329
8,content_cd3d932d4e1c8db0,tier_2_4-10,89332.0,4.0,0.002353,210.198196,206.198196
9,content_306bc78dff1eb683,tier_1_1-3,80821.0,35.0,0.002811,227.187831,192.187831


In [21]:
con.sql(f"""
    SELECT MIN(content_updated_date), MAX(content_updated_date),
           MIN(content_created_date), MAX(content_created_date)
    FROM read_parquet('{rel}/dim_content.parquet')
""").df()

,min(content_updated_date),max(content_updated_date),min(content_created_date),max(content_created_date)
0,2024-10-28,2026-07-06,2024-10-16,2026-07-06


In [22]:
con.sql(f"""
    SELECT MIN(keyword_created_date), MAX(keyword_created_date)
    FROM read_parquet('{rel}/dim_content.parquet')
""").df()

,min(keyword_created_date),max(keyword_created_date)
0,2024-10-16,2026-07-02


In [23]:
con.sql(f"""
    SELECT
        CASE WHEN content_updated_date >= '2026-04-01' THEN 'updated_after_march'
             ELSE 'updated_march_or_before' END AS update_bucket,
        AVG(search_volume) AS avg_search_volume,
        COUNT(*) AS n
    FROM read_parquet('{rel}/dim_content.parquet')
    GROUP BY 1
""").df()

,update_bucket,avg_search_volume,n
0,updated_after_march,228.338082,382739
1,updated_march_or_before,98.364478,136867


In [24]:
con.sql(f"""
    SELECT
        CASE WHEN content_updated_date >= '2026-04-01' THEN 'updated_after_march'
             ELSE 'updated_march_or_before' END AS update_bucket,
        main_intent,
        COUNT(*) AS n
    FROM read_parquet('{rel}/dim_content.parquet')
    GROUP BY 1, 2
    ORDER BY 1, 3 DESC
""").df()

,update_bucket,main_intent,n
0,updated_after_march,informational,227366
1,updated_after_march,None,66116
2,updated_after_march,transactional,45680
3,updated_after_march,commercial,42008
4,updated_after_march,navigational,1569
5,updated_march_or_before,None,82282
6,updated_march_or_before,informational,32867
7,updated_march_or_before,commercial,10754
8,updated_march_or_before,transactional,10276
9,updated_march_or_before,navigational,688


In [25]:
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{rel}/dim_clients.parquet')").df()

,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,is_active,BOOLEAN,YES,None,None,None
2,has_gsc_access,BOOLEAN,YES,None,None,None
3,has_ga4_access,BOOLEAN,YES,None,None,None
4,access_profile,VARCHAR,YES,None,None,None
5,client_created_date,DATE,YES,None,None,None
6,client_updated_date,DATE,YES,None,None,None
7,gsc_data_start,DATE,YES,None,None,None
8,ga4_data_start,DATE,YES,None,None,None


In [26]:
con.sql(f"""
    SELECT
        CASE WHEN content_updated_date >= '2026-04-01' THEN 'updated_after_march'
             ELSE 'updated_march_or_before' END AS update_bucket,
        AVG(backlinks) AS avg_backlinks,
        AVG(word_count) AS avg_word_count,
        AVG(keyword_char_count) AS avg_keyword_chars,
        AVG(keyword_token_count) AS avg_keyword_tokens,
        COUNT(*) AS n
    FROM read_parquet('{rel}/dim_content.parquet')
    GROUP BY 1
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,update_bucket,avg_backlinks,avg_word_count,avg_keyword_chars,avg_keyword_tokens,n
0,updated_after_march,475.440779,2459.604182,30.293197,5.324971,382739
1,updated_march_or_before,323.625117,2537.595634,32.594811,4.178743,136867


## 2. Field classification
Tables: The primary table is fact_content_daily_performance, which is daily grain and gives clicks, impressions, and position on a window I can control. It is joined by dim_content for feature metadata, confirmed to have zero performance columns, so it's safe from leakage. For context, I use dim_clients, confirmed to hold only account-level info (access flags, created/updated dates) — nothing that explains page-level CTR. I excluded fact_content_query_90d because it has a fixed window of April 2–June 30 that overlaps our sealed test month — and even for a decision date inside that window, its clicks_90d figure sums across the entire fixed span through June 30, so it includes clicks from after that date regardless.

Label (lost_clicks): First, I bucket each page into a position tier by its impression-weighted average position. Then, I compute expected CTR as the impression-weighted median CTR for each tier. Then, I compute expected clicks = tier's expected CTR × page's impressions. Finally, I compute lost clicks by subtracting expected clicks − actual clicks. If the value becomes negative, it means the page overperforms.

Exclusion: gsc_avg_position = 0 is treated as "no real position," not rank #1 — confirmed by two checks: CTR-by-position-bucket showed the 0 bucket's CTR sits below both the 1-3 and 4-10 buckets, and the curriculum's own guide filters confirm 0 is meant to be excluded from real position ranges.

## 3. Verify every claim with queries (grain, counts, missingness, windows)

- Grain check — grouping the final rolled-up table by content_hash_id and looking for any page appearing more than once returned an empty result, confirming one row truly equals one page.
- Row count + date span — the March slice contains 9,841,378 rows, spanning 2026-03-01 to 2026-03-31 with no gaps.
- Availability, IS TRUE — of 9,841,378 total rows, 3,611,061 survive the gsc_data_available IS TRUE filter.


- search_volume — a keyword-level metric with a plausible link to CTR (competitive/high-volume queries have crowded results pages that can suppress CTR at a given position). Limitation: no timestamp of its own; an observed correlation with content_updated_date (higher search_volume for recently-updated pages) means I can't fully rule out mild snapshot-timing risk.
- competition_level — same mechanism and same stated limitation as search_volume.
- content_type — a fixed classification, not dependent on anything that happens after page creation.
- content_updated_date, capped — only counted as "updated" if the date falls on or before March 31; updates after that are treated as not-yet-happened, so the feature never uses information from the future. Limitation: 82.6% of rows have no confirmed update before the cutoff, so the feature is NULL for the large majority of pages.
- word_count — the cleanest of the candidates checked for update-correlation (~3% gap between update buckets, smallest of everything tested), though its mechanistic link to CTR specifically (rather than post-click engagement) is weaker than the others.

In [27]:
con.sql(f"""
    WITH daily AS (
        SELECT content_hash_id, gsc_avg_position, gsc_impressions, gsc_clicks
        FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
        WHERE gsc_data_available = TRUE AND gsc_avg_position > 0 AND gsc_impressions > 0
    ),
    rollup AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS impressions
        FROM daily GROUP BY content_hash_id
        HAVING SUM(gsc_impressions) >= 50
    )
    SELECT content_hash_id, COUNT(*) AS n
    FROM rollup
    GROUP BY content_hash_id
    HAVING COUNT(*) > 1
""").df()

,content_hash_id,n


In [28]:
con.sql(f"""
    SELECT COUNT(*) AS n_rows, MIN(report_date) AS earliest, MAX(report_date) AS latest
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
""").df()

,n_rows,earliest,latest
0,9841378,2026-03-01,2026-03-31


In [29]:
con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS rows_survive_availability
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
""").df()

,total_rows,rows_survive_availability
0,9841378,3611061.0


In [30]:
# Final feature frame — one query, includes everything needed downstream
df = con.sql(f"""
    WITH daily AS (
        SELECT content_hash_id, gsc_avg_position, gsc_impressions, gsc_clicks
        FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
        WHERE gsc_data_available = TRUE AND gsc_avg_position > 0 AND gsc_impressions > 0
    ),
    rollup AS (
        SELECT
            content_hash_id,
            SUM(gsc_impressions) AS impressions,
            SUM(gsc_clicks) AS clicks,
            SUM(gsc_avg_position * gsc_impressions) / SUM(gsc_impressions) AS weighted_avg_position
        FROM daily
        GROUP BY content_hash_id
        HAVING SUM(gsc_impressions) >= 50
    ),
    tiered AS (
        SELECT *,
            CASE
                WHEN weighted_avg_position <= 3 THEN 'tier_1_1-3'
                WHEN weighted_avg_position <= 10 THEN 'tier_2_4-10'
                WHEN weighted_avg_position <= 20 THEN 'tier_3_11-20'
                ELSE 'tier_4_21plus'
            END AS position_tier
        FROM rollup
    ),
    benchmark(position_tier, expected_ctr) AS (
        VALUES
            ('tier_1_1-3', 0.002811), ('tier_2_4-10', 0.002353),
            ('tier_3_11-20', 0.002226), ('tier_4_21plus', 0.000748)
    ),
    labeled AS (
        SELECT
            t.content_hash_id, t.impressions, t.clicks, t.weighted_avg_position,
            b.expected_ctr,
            (b.expected_ctr * t.impressions) - t.clicks AS lost_clicks
        FROM tiered t
        JOIN benchmark b USING (position_tier)
    )
    SELECT
        l.content_hash_id, l.lost_clicks, l.impressions, l.clicks,
        l.weighted_avg_position, l.expected_ctr,
        c.search_volume, c.competition_level, c.content_type,
        CASE WHEN c.content_updated_date <= DATE '2026-03-31'
             THEN DATE_DIFF('day', c.content_updated_date, DATE '2026-03-31')
             ELSE NULL END AS days_since_update_capped,
        c.word_count
    FROM labeled l
    JOIN read_parquet('{rel}/dim_content.parquet') c
        ON l.content_hash_id = c.content_hash_id
""").df()

df.columns.tolist()  # sanity check — should show all 11 columns, including 'clicks'

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

['content_hash_id',
 'lost_clicks',
 'impressions',
 'clicks',
 'weighted_avg_position',
 'expected_ctr',
 'search_volume',
 'competition_level',
 'content_type',
 'days_since_update_capped',
 'word_count']

In [31]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

model_df = df.copy()
model_df['has_word_count'] = model_df['word_count'].notna().astype(int)
model_df['word_count'] = model_df['word_count'].fillna(0)
model_df['updated_before_march31'] = model_df['days_since_update_capped'].notna().astype(int)
model_df['days_since_update_capped'] = model_df['days_since_update_capped'].fillna(0)
model_df['search_volume'] = model_df['search_volume'].fillna(model_df['search_volume'].median())
model_df = pd.get_dummies(model_df, columns=['content_type', 'competition_level'], drop_first=True)
model_df['expected_clicks'] = model_df['expected_ctr'] * model_df['impressions']

def fit_and_score(feature_cols, label='honest'):
    X = model_df[feature_cols]
    y = model_df['lost_clicks']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model = LinearRegression().fit(X_train, y_train)
    score = model.score(X_test, y_test)
    print(f"{label}: R² = {score:.4f}")
    return score

# All FIVE honest features this time, including search_volume
honest_cols = ['search_volume', 'word_count', 'has_word_count',
               'days_since_update_capped', 'updated_before_march31'] + \
              [c for c in model_df.columns if c.startswith('content_type_') or c.startswith('competition_level_')]

fit_and_score(honest_cols, 'Honest (5 features)')
fit_and_score(honest_cols + ['expected_clicks', 'clicks'], 'Leaked (expected_clicks + clicks added)')

Honest (5 features): R² = 0.0072
Leaked (expected_clicks + clicks added): R² = 1.0000


1.0

In [32]:
fit_and_score(honest_cols + ['expected_clicks', 'clicks'], 'Leaked (expected_clicks + clicks added)')

Leaked (expected_clicks + clicks added): R² = 1.0000


1.0

**Leak trap:** Adding `expected_clicks` and `clicks` — both direct components of the lost_clicks
formula (`lost_clicks = expected_clicks − clicks`) — pushed R² from 0.0072 to 1.0000, since the
model could reconstruct the label exactly from its own ingredients. Both columns are removed from
the feature set.

**R² = 0.0072, using only the five honest features, is the number I'm keeping.**
This is a low but honest result — it means page metadata alone explains almost none of the
variance in lost_clicks, which is a real finding about this feature set, not a failure of the
notebook.

## 4. Data limits

- Content updated date is null 82.6% of the time, so the capped update-date feature only has a real value for a minority of pages updated on or before March 31.
- Word count missingness follows content type that has 0% missing for comparison article, but it has 31.6% missing for keyword article, so if I fill them with 0, it would have made it a false signal
- Search volume and competition level have timing risk because they don't have a timestamp on their own and I cannot determine whether it is safe which means high-value pages get prioritized for updates, or it recomputes the value after it got edited.

## Self-check

Before you submit, confirm each line honestly:

- [DONE] Every section above is filled — markdown thinking AND the code that backs it
- [DONE] The notebook runs top to bottom with no errors (Runtime → Run all)
- [DONE] No client names, URLs, or private queries anywhere
- [DONE] My claims use careful words: observed, measured, directional, decision-support
- [DONE] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.